In [0]:

%sql
-- Creating Valume for raw data
CREATE VOLUME IF NOT EXISTS dbr_dev.default.raw_data;

In [0]:
import requests
import zipfile
import io
import os
import pyspark.sql.functions as F

In [0]:
# Is Executed importing the CSV Data of "Poland Houses Prices" from Kaggle via API KEY and Kaggle API
volume_path = "/Volumes/dbr_dev/default/raw_data"
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/dawidcegielski/house-prices-in-poland"
kaggle_token = "KGAT_a0356ab89cb4776324bbedf2b57ce23a"
# (CHANGEDKEYFOR_)GAT_(_PUSHON_)a0356ab89cb4776324bbedf2b57ce23a(_GIHAB)
print("Connect to Kaggle REST API")
headers = {"Authorization": f"Bearer {kaggle_token}"}
response = requests.get(dataset_url, headers=headers, stream=True)

if response.status_code == 200:
    print("Data is downloaded")
    print("Started unziping")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            target_path = os.path.join(volume_path, file_name)
            with open(target_path, "wb") as f:
                f.write(z.read(file_name))
            print(f"File: {file_name} copleated")
            
    print(f"Done!")
else:
    print(f"Download error: {response.status_code}")
    print(response.text)



In [0]:
# Is Executed a transformation RAW_DATA to Bronze Table
file_path = "/Volumes/dbr_dev/default/raw_data/Houses.csv"
df = spark.read.csv(file_path, header=True,encoding="windows-1250")
df.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.default.Artem_Zharkov_PolHousePrices_Bronze")   

In [0]:
file_path = "dbr_dev.default.Artem_Zharkov_PolHousePrices_Bronze"
bronze_df = spark.read.table(file_path).na.replace({"": None, "N/A": None})
display(bronze_df.limit(10))

In [0]:
# Transformation to Silver layer
silver_df = (
    bronze_df.dropDuplicates()
    .withColumnRenamed("id", "house_id")
    .withColumnRenamed("_c0", "id")
    .withColumn("house_id", F.col("house_id").cast("double").cast("integer"))
    .withColumn("id", F.col("id").cast("integer"))
    .withColumn("address",F.initcap(F.lower(F.trim(F.col("address")))))
    .withColumn("city", F.initcap(F.lower(F.trim(F.col("city")))))
    .withColumn("floor",F.col("floor").cast("double").cast("integer"))
    .withColumn("latitude",F.round(F.col("latitude").cast("double"),7))
    .withColumn("longitude",F.round(F.col("longitude").cast("double"),7))
    .withColumn("price",F.round(F.col("price").cast("double"),2))
    .withColumn("currency_type", F.lit("PLN"))
    .withColumn("rooms",F.col("rooms").cast("double").cast("integer"))

    # ?
    .withColumn("sq",F.round(F.col("sq").cast("double"),2).cast("double"))
    .withColumn("year",F.col("year").cast("double").cast("integer"))
    .withColumn("year", F.year(F.try_to_date(F.col("year").cast("string"), "yyyy")))
)
display(silver_df.limit(10))

# Is Executed a transformation Bronze Table to Silver Table
# option("overwriteSchema", "true") 
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbr_dev.default.Artem_Zharkov_PolHousePrices_Silver")